In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import json
load_dotenv()
client = OpenAI()

def classifiy_email(prompt):
    response = client.chat.completions.create(
        model = "gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content




with open("test_emails.json","r") as f:
    emails = json.load(f)

In [2]:
email = emails[0]
print(email)
subject = email['subject']
body = email['body']

{'id': 1, 'subject': 'Invoice Discrepancy', 'sender': 'finance@company.com', 'body': 'My invoice is wrong, please fix it.', 'correct_label': 'Billing', 'urgency': 'High', 'difficulty': 'easy', 'why_tricky': 'The request is straightforward and specifically asks for a billing issue to be resolved.'}


In [3]:
prompt = f"""
# ROLE
You are an expert support email classifier for a B2B SaaS company.
You have 10 years of experience triaging customer support emails.

# TASK  
Classify the incoming email into exactly one primary category,
one urgency level, and one sub-category where applicable.

# CATEGORIES
Primary (pick ONE):
- Billing
- Technical
- Feature Request
- Spam
- Other

Urgency (pick ONE):
- High    (customer cannot use the product)
- Medium  (issue exists but workaround available)
- Low     (question or minor inconvenience)

Billing Sub-category (only if Primary = Billing):
- Refund Request
- Failed Payment
- Pricing Question
- Invoice Issue
- Not Applicable

# RULES
- RETURN ONLY valid JSON. No explanation. No markdown.
- If the email is in any language other than English, still classify it.
- If you cannot determine the category, use "Other" and urgency "Low".

# OUTPUT FORMAT
Return ONLY this JSON structure, nothing else:
{{
  "category": "[Primary Category]",
  "urgency": "[Urgency Level]",
  "billing_sub": "[Sub-category or Not Applicable]"
}}

# EMAIL TO CLASSIFY
Subject: {subject}
Body: {body}
"""

In [4]:
output = classifiy_email(prompt)

In [6]:
print(output)

{
  "category": "Billing",
  "urgency": "Medium",
  "billing_sub": "Invoice Issue"
}


## Few short prompting

In [8]:
examples = [
        {
            "subject": "Charged $299 instead of $99",
            "body": "I upgraded to the Pro plan but you charged me $299, the Enterprise price.",
            "output": {"category": "Billing", "urgency": "High", "billing_sub": "Failed Payment"}
        },
        {
            "subject": "SFDC integration not syncing",
            "body": "Our Salesforce integration stopped syncing leads 3 days ago. Last sync: Nov 12.",
            "output": {"category": "Technical", "urgency": "High", "billing_sub": "Not Applicable"}
        },
        {
            "subject": "Can you add Slack notifications?",
            "body": "Would love to get a Slack ping when a new lead comes in. Is this planned?",
            "output": {"category": "Feature Request", "urgency": "Low", "billing_sub": "Not Applicable"}
        },
        {
            "subject": "WIN $500 AMAZON GIFT CARD!!!",
            "body": "Click here to claim your prize NOW limited time offer!!!!",
            "output": {"category": "Spam", "urgency": "Low", "billing_sub": "Not Applicable"}
        },
    ]
    


In [9]:
    # Build the examples block
examples_block = ""
for i, ex in enumerate(examples, 1):
    examples_block += f"""
EXAMPLE {i}:
Subject: {ex['subject']}
Body: {ex['body']}
Output: {ex['output']}
"""

In [11]:
print(examples_block)


EXAMPLE 1:
Subject: Charged $299 instead of $99
Body: I upgraded to the Pro plan but you charged me $299, the Enterprise price.
Output: {'category': 'Billing', 'urgency': 'High', 'billing_sub': 'Failed Payment'}

EXAMPLE 2:
Subject: SFDC integration not syncing
Body: Our Salesforce integration stopped syncing leads 3 days ago. Last sync: Nov 12.
Output: {'category': 'Technical', 'urgency': 'High', 'billing_sub': 'Not Applicable'}

EXAMPLE 3:
Subject: Can you add Slack notifications?
Body: Would love to get a Slack ping when a new lead comes in. Is this planned?
Output: {'category': 'Feature Request', 'urgency': 'Low', 'billing_sub': 'Not Applicable'}

EXAMPLE 4:
Subject: WIN $500 AMAZON GIFT CARD!!!
Body: Click here to claim your prize NOW limited time offer!!!!
Output: {'category': 'Spam', 'urgency': 'Low', 'billing_sub': 'Not Applicable'}



In [12]:
prompt =f"""
CRITICAL: Return ONLY valid JSON. No explanation. No markdown.

# ROLE
You are a support email classifier for a B2B SaaS company with deep 
knowledge of our product and customer patterns.

# EXAMPLES OF CORRECT CLASSIFICATIONS
Study these examples carefully — they reflect OUR specific business context:
{examples_block}

# NOW CLASSIFY THIS EMAIL
Subject: {subject}
Body: {body}

REMINDER: Return ONLY the JSON. Exactly 3 fields: category, urgency, billing_sub.
"""

In [14]:
print(prompt)


CRITICAL: Return ONLY valid JSON. No explanation. No markdown.

# ROLE
You are a support email classifier for a B2B SaaS company with deep 
knowledge of our product and customer patterns.

# EXAMPLES OF CORRECT CLASSIFICATIONS
Study these examples carefully — they reflect OUR specific business context:

EXAMPLE 1:
Subject: Charged $299 instead of $99
Body: I upgraded to the Pro plan but you charged me $299, the Enterprise price.
Output: {'category': 'Billing', 'urgency': 'High', 'billing_sub': 'Failed Payment'}

EXAMPLE 2:
Subject: SFDC integration not syncing
Body: Our Salesforce integration stopped syncing leads 3 days ago. Last sync: Nov 12.
Output: {'category': 'Technical', 'urgency': 'High', 'billing_sub': 'Not Applicable'}

EXAMPLE 3:
Subject: Can you add Slack notifications?
Body: Would love to get a Slack ping when a new lead comes in. Is this planned?
Output: {'category': 'Feature Request', 'urgency': 'Low', 'billing_sub': 'Not Applicable'}

EXAMPLE 4:
Subject: WIN $500 AMAZO

In [15]:
classification_output = classifiy_email(prompt)
classification_output

'{"category": "Billing", "urgency": "High", "billing_sub": "Invoice Error"}'

## Dyanmic Prompting

In [16]:
PRODUCT_CONFIGS = {
    "saas_crm": {
        "product_name": "CRM Platform",
        "categories": ["Billing", "Technical", "Feature Request", "Spam", "Other"],
        "urgency_levels": ["High", "Medium", "Low"],
        "sla_hours": {"High": 2, "Medium": 8, "Low": 24},
        "examples": [
            {"subject": "SFDC not syncing", "body": "Salesforce stopped pulling leads",
             "output": {"category": "Technical", "urgency": "High"}}
        ]
    },
    "fintech_payments": {
        "product_name": "Payment Gateway",
        "categories": ["Transaction Failed", "Fraud Alert", "Settlement Delay", "KYC Issue", "Other"],
        "urgency_levels": ["Critical", "High", "Medium"],
        "sla_hours": {"Critical": 1, "High": 4, "Medium": 12},
        "examples": [
            {"subject": "Payment declined at checkout", "body": "Card declined even though sufficient balance",
             "output": {"category": "Transaction Failed", "urgency": "High"}}
        ]
    }
}

In [24]:
# Dynamic Prompting
# run for fintech
config = PRODUCT_CONFIGS["saas_crm"]
config

{'product_name': 'CRM Platform',
 'categories': ['Billing', 'Technical', 'Feature Request', 'Spam', 'Other'],
 'urgency_levels': ['High', 'Medium', 'Low'],
 'sla_hours': {'High': 2, 'Medium': 8, 'Low': 24},
 'examples': [{'subject': 'SFDC not syncing',
   'body': 'Salesforce stopped pulling leads',
   'output': {'category': 'Technical', 'urgency': 'High'}}]}

In [25]:
body = "Card declined even though sufficient balance"

In [26]:
prompt = f"""CRITICAL: Return ONLY valid JSON.

# ROLE
You are an expert support email classifier for {config['product_name']}.

# CATEGORIES
Classify into exactly one of:
{config['categories']}

# URGENCY LEVELS  
Use exactly one of: {', '.join(config['urgency_levels'])}
SLAs: {config['sla_hours']}

# EXAMPLES FROM THIS PRODUCT'S HISTORY
{config['examples']}

# EMAIL TO CLASSIFY
Body: {body}

REMINDER: Return ONLY JSON with keys "category" and "urgency"."""

In [27]:
output = classifiy_email(prompt)
print(output)

```json
{
  "category": "Billing",
  "urgency": "High"
}
```
